# Notebook 02: Paper Figures

Reproduce and inspect all 8 paper figures interactively.
Run the full pipeline (`scripts/run_all.py`) before using this notebook.

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from transformer_lol.utils import get_project_root
from transformer_lol.plotting import (
    fig01_system_schematic, fig02_annex_g_validation,
    fig03_daily_trajectories, fig04_annual_lol_boxplots,
    fig05_pv_lol_nonmonotonic, fig06_sensitivity_heatmap,
    fig07_surrogate_accuracy, fig08_runtime_comparison,
)

root = get_project_root()
fig_dir = root / 'results' / 'figures'
tab_dir = root / 'results' / 'tables'

# Load MC results
mc_df = pd.read_parquet(tab_dir / 'mc_results_all_scenarios.parquet')
print(f'MC results: {len(mc_df):,} rows, {mc_df["scenario_name"].nunique()} scenarios')

# Load metrics
with open(root / 'results' / 'metrics.json') as f:
    metrics = json.load(f)
print('\nKey metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v}')

In [ ]:
# Core finding: non-monotonic LoL vs PV penetration
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150

fig05_pv_lol_nonmonotonic(fig_dir, mc_df)
plt.show()

In [ ]:
# Summary statistics table
stats = mc_df.groupby(['pv_penetration', 'ambient_delta_c'])['lol_percent_annual'].agg(
    mean='mean', std='std', p5=lambda x: x.quantile(0.05), p95=lambda x: x.quantile(0.95)
).round(4)
print(stats.to_string())